In [1]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 1000)

In [2]:
url = "https://raw.githubusercontent.com/J-SNACKKB/FLIP/9f58a97455d4133669c04b0a723f9d6e49e29eb2/splits/gb1/four_mutations_full_data.csv.zip"

gb1 = pd.read_csv(url)
gb1.columns = [col.strip() for col in gb1.columns]
gb1.columns = gb1.columns.str.replace(' ', '_')
gb1.columns = gb1.columns.str.lower()
gb1 = gb1.drop(columns=["sequence"])
print(gb1.head())

  variants  hd  count_input  count_selected   fitness   keep one_vs_rest one_vs_rest_validation two_vs_rest two_vs_rest_validation three_vs_rest three_vs_rest_validation sampled sampled_validation low_vs_high low_vs_high_validation
0     VDGV   0        92735          338346  1.000000   True       train                    NaN       train                    NaN         train                      NaN   train                NaN        test                    NaN
1     ADGV   1           34              43  0.061910  False         NaN                    NaN         NaN                    NaN           NaN                      NaN     NaN                NaN         NaN                    NaN
2     CDGV   1          850             641  0.242237  False         NaN                    NaN         NaN                    NaN           NaN                      NaN     NaN                NaN         NaN                    NaN
3     DDGV   1           63              63  0.006472  False         NaN

/var/folders/th/n8tc1q8948g1bhbgbwrxyhbm0000gn/T/ipykernel_91209/968537127.py:3: DtypeWarning: Columns (8,10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  gb1 = pd.read_csv(url)


In [3]:

data = gb1.copy()

# Keep valid four-letter variants with numerical fitness
data.fillna(0, inplace=True)

data["variants"] = data["variants"].astype(str).str.strip()
data["fitness"] = pd.to_numeric(data["fitness"], errors="coerce")
data["hd"] = data["hd"].astype(int)




data = data.dropna()
data = data[data["variants"].str.len() == 4].copy()

# Split the four-letter variant into four categorical variables
for i in range(4):
    data[f"p{i+1}"] = data["variants"].str[i]





data = data.query("variants != 'VDGV'").copy().reset_index(drop=True)  # Remove the wild-type variant



print(data.head())
print("Number of observations:", len(data))

  variants  hd  count_input  count_selected   fitness   keep one_vs_rest one_vs_rest_validation two_vs_rest two_vs_rest_validation three_vs_rest three_vs_rest_validation sampled sampled_validation low_vs_high low_vs_high_validation p1 p2 p3 p4
0     ADGV   1           34              43  0.061910  False           0                      0           0                      0             0                        0       0                  0           0                      0  A  D  G  V
1     CDGV   1          850             641  0.242237  False           0                      0           0                      0             0                        0       0                  0           0                      0  C  D  G  V
2     DDGV   1           63              63  0.006472  False           0                      0           0                      0             0                        0       0                  0           0                      0  D  D  G  V
3     EDGV   1          

In [4]:
# prefix tree
landscape = data[["variants", "fitness"]].copy()

landscape["variants"] = landscape["variants"].astype(str).str.strip()
landscape["fitness"] = pd.to_numeric(
    landscape["fitness"], errors="coerce"
)

landscape = landscape.dropna()
landscape = landscape[landscape["variants"].str.len() == 4]

prefix_tables = []

for depth in range(1, 5):

    tmp = landscape.copy()
    tmp["prefix"] = tmp["variants"].str[:depth]

    stats = (
        tmp.groupby("prefix")["fitness"]
        .agg(
            count="count",
            minimum="min",
            maximum="max",
            mean="mean",
            median="median",
            std="std",
            nonbinding=lambda x: (x == 0).mean(),
            better_than_wt=lambda x: (x > 1).mean()
        )
        .reset_index()
    )

    stats["depth"] = depth
    prefix_tables.append(stats)

prefix_stats = pd.concat(prefix_tables, ignore_index=True)

prefix_stats = prefix_stats[
    [
        "depth", "prefix", "count",
        "minimum", "maximum", "mean", "median", "std",
        "nonbinding", "better_than_wt"
    ]
]

print(prefix_stats.head(20))

    depth prefix  count  minimum   maximum      mean    median       std  nonbinding  better_than_wt
0       1      A   4964      0.0  7.556869  0.091099  0.000000  0.420733    0.560637        0.027599
1       1      C   7660      0.0  4.470334  0.079954  0.009669  0.324150    0.031332        0.020757
2       1      D   6929      0.0  3.489668  0.010078  0.000000  0.093237    0.538173        0.001443
3       1      E   7430      0.0  4.990726  0.062753  0.002947  0.324713    0.297981        0.019650
4       1      F   7512      0.0  8.761966  0.154545  0.004249  0.609254    0.285809        0.048855
5       1      G   7604      0.0  2.438274  0.009988  0.001474  0.070691    0.336796        0.001315
6       1      H   7655      0.0  4.720571  0.056509  0.003256  0.287322    0.159373        0.017374
7       1      I   7682      0.0  6.042078  0.140826  0.004369  0.533047    0.115204        0.046082
8       1      K   7731      0.0  4.574229  0.036675  0.002909  0.221505    0.126116       

In [5]:
trie = {}

for _, row in prefix_stats.iterrows():

    prefix = row["prefix"]
    node = trie

    for aa in prefix:
        node = node.setdefault(aa, {})

    node["_stats"] = {
        "count": int(row["count"]),
        "min": float(row["minimum"]),
        "max": float(row["maximum"]),
        "mean": float(row["mean"]),
        "median": float(row["median"]),
        "std": None if pd.isna(row["std"]) else float(row["std"]),
        "p_nonbinding": float(row["nonbinding"]),
        "p_better_than_wt": float(row["better_than_wt"])
    }

In [6]:
display_stats = prefix_stats.copy()

display_stats["nonbinding_%"] = (
    100 * display_stats["nonbinding"]
)

display_stats["better_than_wt_%"] = (
    100 * display_stats["better_than_wt"]
)

print(
    display_stats[
        ["depth", "prefix", "count", "minimum", "maximum",
         "mean", "nonbinding_%", "better_than_wt_%"]
    ].round(2)
)

        depth prefix  count  minimum  maximum  mean  nonbinding_%  better_than_wt_%
0           1      A   4964     0.00     7.56  0.09         56.06              2.76
1           1      C   7660     0.00     4.47  0.08          3.13              2.08
2           1      D   6929     0.00     3.49  0.01         53.82              0.14
3           1      E   7430     0.00     4.99  0.06         29.80              1.97
4           1      F   7512     0.00     8.76  0.15         28.58              4.89
...       ...    ...    ...      ...      ...   ...           ...               ...
157756      4   YYYS      1     0.00     0.00  0.00          0.00              0.00
157757      4   YYYT      1     0.02     0.02  0.02          0.00              0.00
157758      4   YYYV      1     0.04     0.04  0.04          0.00              0.00
157759      4   YYYW      1     0.01     0.01  0.01          0.00              0.00
157760      4   YYYY      1     0.01     0.01  0.01          0.00           

In [7]:
print(prefix_stats[prefix_stats["prefix"] == "N"])
print(prefix_stats[prefix_stats["prefix"].str.startswith("N")])
print(prefix_stats[prefix_stats["depth"] == 2].sort_values("prefix"))

    depth prefix  count  minimum   maximum      mean    median       std  nonbinding  better_than_wt
11      1      N   7599      0.0  3.061371  0.014004  0.002586  0.099949    0.273589        0.002106
       depth prefix  count   minimum   maximum      mean    median       std  nonbinding  better_than_wt
11         1      N   7599  0.000000  3.061371  0.014004  0.002586  0.099949    0.273589        0.002106
240        2     NA    381  0.000000  1.097019  0.013675  0.002885  0.077748    0.191601        0.002625
241        2     NC    366  0.000000  0.743347  0.020650  0.008841  0.062473    0.292350        0.000000
242        2     ND    380  0.000000  1.115236  0.008993  0.002610  0.060251    0.276316        0.002632
243        2     NE    380  0.000000  0.296611  0.005285  0.002366  0.019512    0.284211        0.000000
...      ...    ...    ...       ...       ...       ...       ...       ...         ...             ...
96595      4   NYYS      1  0.013535  0.013535  0.013535  0.013

In [8]:
# Experiment 2



wild_type = "VDGV"

# Work from all measured variants
landscape = data[["variants", "fitness"]].copy()
landscape["variants"] = landscape["variants"].astype(str).str.strip()
landscape["fitness"] = pd.to_numeric(
    landscape["fitness"], errors="coerce"
)

landscape = landscape.dropna()
landscape = landscape[landscape["variants"].str.len() == 4].copy()

# Hamming distance from wild type
landscape["HD"] = landscape["variants"].apply(
    lambda s: sum(a != b for a, b in zip(s, wild_type))
)

# Fitness distribution by mutation count
fitness_by_hd = landscape.groupby("HD")["fitness"].agg(
    count="count",
    minimum="min",
    q05=lambda x: x.quantile(0.05),
    q25=lambda x: x.quantile(0.25),
    median="median",
    mean="mean",
    q75=lambda x: x.quantile(0.75),
    q95=lambda x: x.quantile(0.95),
    maximum="max",
    std="std",
).reset_index()

fitness_by_hd["nonbinding"] = landscape.groupby("HD")["fitness"].apply(
    lambda x: (x == 0).mean()
).values

fitness_by_hd["better_than_wt"] = landscape.groupby("HD")["fitness"].apply(
    lambda x: (x > 1).mean()
).values

print(fitness_by_hd)


   HD   count   minimum       q05       q25    median      mean       q75       q95   maximum       std  nonbinding  better_than_wt
0   1      76  0.003424  0.004305  0.011320  0.122439  0.640472  1.098620  2.564570  3.901461  0.934226    0.000000        0.263158
1   2    2091  0.000000  0.000000  0.003273  0.018668  0.295653  0.220321  1.638600  5.390733  0.661240    0.128647        0.094692
2   3   26019  0.000000  0.000000  0.001877  0.004792  0.153056  0.024869  0.975000  6.274946  0.512061    0.156770        0.048964
3   4  121174  0.000000  0.000000  0.001252  0.003169  0.060861  0.007613  0.124583  8.761966  0.354683    0.207379        0.017751


In [9]:


variable_positions = [39, 40, 41, 54]

def get_mutations(sequence, wild_type="VDGV"):
    return [
        f"{wt}{position}{aa}"
        for wt, position, aa in zip(
            wild_type, variable_positions, sequence
        )
        if wt != aa
    ]



landscape["mutations"] = landscape["variants"].apply(get_mutations)

print(landscape[["variants", "HD", "mutations", "fitness"]].head(10))


  variants  HD mutations   fitness
0     ADGV   1    [V39A]  0.061910
1     CDGV   1    [V39C]  0.242237
2     DDGV   1    [V39D]  0.006472
3     EDGV   1    [V39E]  0.032719
4     FDGV   1    [V39F]  0.377101
5     GDGV   1    [V39G]  0.008298
6     HDGV   1    [V39H]  0.026480
7     IDGV   1    [V39I]  1.445905
8     KDGV   1    [V39K]  0.020161
9     LDGV   1    [V39L]  1.690164


In [10]:

def fitness_category(f):
    if f == 0:
        return "nonbinding"
    elif f < 0.5:
        return "low"
    elif f < 1:
        return "below_wt"
    elif f < 2:
        return "at_or_above_wt"
    else:
        return "high"

landscape["fitness_class"] = landscape["fitness"].apply(fitness_category)

print(landscape["fitness_class"].value_counts())


fitness_class
low               114062
nonbinding         29477
below_wt            2178
at_or_above_wt      1952
high                1691
Name: count, dtype: int64


In [11]:

mutation_rows = []

for _, row in landscape.iterrows():
    for mutation in row["mutations"]:
        mutation_rows.append({
            "mutation": mutation,
            "fitness": row["fitness"],
            "fitness_class": row["fitness_class"]
        })

mutation_data = pd.DataFrame(mutation_rows)

mutation_stats = mutation_data.groupby("mutation").agg(
    support=("fitness", "count"),
    mean_fitness=("fitness", "mean"),
    median_fitness=("fitness", "median"),
    nonbinding=("fitness_class", lambda x: (x == "nonbinding").mean()),
    high_fitness=("fitness_class", lambda x: (x == "high").mean()),
    better_than_wt=("fitness", lambda x: (x > 1).mean())
).reset_index()

print(mutation_stats.sort_values(
    ["support", "mean_fitness"],
    ascending=[False, False]
)
)


   mutation  support  mean_fitness  median_fitness  nonbinding  high_fitness  better_than_wt
72     V54S     7957      0.073707        0.004331    0.089984      0.005907        0.019982
66     V54L     7955      0.053080        0.003320    0.108737      0.005908        0.012948
69     V54P     7955      0.011628        0.003150    0.109239      0.000000        0.000754
71     V54R     7943      0.003481        0.002523    0.139746      0.000000        0.000000
73     V54T     7937      0.137650        0.005212    0.092604      0.014363        0.045105
57     V54A     7928      0.377548        0.009015    0.079970      0.073032        0.117810
55     V39W     7877      0.191274        0.004642    0.024756      0.032500        0.057509
52     V39R     7854      0.030066        0.002873    0.044818      0.002165        0.007257
63     V54H     7808      0.010049        0.002530    0.227203      0.000256        0.000640
62     V54G     7788      0.353403        0.009372    0.125321      0.

In [12]:

global_better = (landscape["fitness"] > 1).mean()
global_high = (landscape["fitness_class"] == "high").mean()
global_nonbinding = (landscape["fitness"] == 0).mean()

mutation_stats["lift_better_than_wt"] = (
    mutation_stats["better_than_wt"] / global_better
)

mutation_stats["lift_high"] = (
    mutation_stats["high_fitness"] / global_high
)

mutation_stats["lift_nonbinding"] = (
    mutation_stats["nonbinding"] / global_nonbinding
)

print(mutation_stats.sort_values(
    "lift_better_than_wt", ascending=False
).head(20))


   mutation  support  mean_fitness  median_fitness  nonbinding  high_fitness  better_than_wt  lift_better_than_wt  lift_high  lift_nonbinding
62     V54G     7788      0.353403        0.009372    0.125321      0.061505        0.121854             4.995919   5.432507         0.635002
57     V54A     7928      0.377548        0.009015    0.079970      0.073032        0.117810             4.830125   6.450682         0.405207
20     G41C     7451      0.281724        0.012650    0.072608      0.045363        0.085358             3.499594   4.006755         0.367903
27     G41L     7704      0.230552        0.006153    0.081906      0.044652        0.076454             3.134542   3.943963         0.415015
23     G41F     7589      0.195912        0.007213    0.107392      0.030439        0.062459             2.560760   2.688550         0.544157
19     G41A     7642      0.204450        0.005491    0.099320      0.034284        0.061895             2.537635   3.028203         0.503252
58    

In [13]:


pair_rows = []

for _, row in landscape.iterrows():
    mutations = row["mutations"]

    for m1, m2 in combinations(mutations, 2):
        pair_rows.append({
            "mutation1": m1,
            "mutation2": m2,
            "fitness": row["fitness"],
            "fitness_class": row["fitness_class"]
        })

pair_data = pd.DataFrame(pair_rows)

pair_stats = pair_data.groupby(
    ["mutation1", "mutation2"]
).agg(
    support=("fitness", "count"),
    mean_fitness=("fitness", "mean"),
    median_fitness=("fitness", "median"),
    nonbinding=("fitness_class", lambda x: (x == "nonbinding").mean()),
    high_fitness=("fitness_class", lambda x: (x == "high").mean()),
    better_than_wt=("fitness", lambda x: (x > 1).mean())
).reset_index()

global_better = (landscape["fitness"] > 1).mean()
global_high = (landscape["fitness_class"] == "high").mean()

pair_stats["lift_better_than_wt"] = (
    pair_stats["better_than_wt"] / global_better
)

pair_stats["lift_high"] = (
    pair_stats["high_fitness"] / global_high
)

pair_stats = pair_stats[pair_stats["support"] >= 30]

print(pair_stats.sort_values(
    "lift_better_than_wt", ascending=False
).head(30))


     mutation1 mutation2  support  mean_fitness  median_fitness  nonbinding  high_fitness  better_than_wt  lift_better_than_wt  lift_high
741       G41C      V54A      397      2.370430        2.263398    0.000000      0.544081        0.707809            29.019568  48.056700
879       G41L      V54G      399      1.745775        1.449703    0.000000      0.416040        0.571429            23.428101  36.747338
803       G41F      V54G      398      1.500941        1.194482    0.005025      0.301508        0.542714            22.250809  26.631086
722       G41A      V54A      399      1.613022        0.997310    0.002506      0.363409        0.498747            20.448211  32.098579
746       G41C      V54G      391      1.219518        0.943819    0.005115      0.240409        0.475703            19.503445  21.234488
890       G41L      V54T      400      0.892761        0.336620    0.000000      0.155000        0.350000            14.349712  13.690597
898       G41M      V54G      393 